# Generative AI 020 — Building an AI Agent (ReAct)

The source's `create_react_agent` / `AgentExecutor` / `hub` code no longer
imports. This notebook writes the ReAct loop by hand, then runs the modern
`create_agent` — a LangGraph graph — end to end. Model decisions are
**scripted**; the tools really run. **No API key.**

| Part | What we check |
|---|---|
| A | which of the source's imports still exist |
| B | the ReAct loop by hand: **770 → 905 → 1,070** characters per call |
| C | `create_agent`: a `CompiledStateGraph`, messages sent **1, 3, 5** |
| D | `recursion_limit`: the default is **9,999** — set your own |

Needs `langchain`, `langgraph`, `langchain-core`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — The source's code, on the installed version

In [ ]:
import importlib
for mod, name in (("langchain.agents", "create_react_agent"),
                  ("langchain.agents", "AgentExecutor"),
                  ("langchain.hub", None),
                  ("langchain.agents", "create_agent")):
    try:
        m = importlib.import_module(mod)
        print(mod, name, "->", "imports" if (name is None or hasattr(m, name)) else "missing")
    except ImportError:
        print(mod, name, "-> ImportError")
# langchain.agents create_react_agent -> missing
# langchain.agents AgentExecutor      -> missing
# langchain.hub None                  -> ImportError
# langchain.agents create_agent       -> imports
#
# The source's agent code does not run on langchain 1.2.15. The source itself
# calls it outdated and points at LangGraph - and create_agent IS LangGraph.

## Part B — ReAct, by hand

In [ ]:
import re
from langchain_core.tools import tool

KB = {"capital of france": "Paris", "population of paris": "2.1 million",
      "capital of madhya pradesh": "Bhopal"}
WEATHER = {"bhopal": {"temperature": 40, "condition": "partly cloudy"}}

@tool
def search(query: str) -> str:
    """Search a small offline encyclopaedia for a fact."""
    return KB.get(query.lower().strip(), "no result")

@tool
def get_weather_data(city: str) -> dict:
    """Fetch the current weather for a city (offline stand-in for a weather API)."""
    return WEATHER.get(city.lower().strip(), {"error": "unknown city"})

TOOLS = {"search": search, "get_weather_data": get_weather_data}

PROMPT = """Answer the following questions as best you can. You have access to the following tools:
{tools}

Use the following format:
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!
Question: {input}
Thought:{agent_scratchpad}"""

def react(question, model_replies, max_iterations=10):
    """The ReAct loop. The model's replies are SCRIPTED; parsing, tool calls and
    the scratchpad are real."""
    scratchpad, sent = "", []
    replies = iter(model_replies)
    tools_text = "\n".join(f"{n}: {t.description}" for n, t in TOOLS.items())
    for _ in range(max_iterations):
        prompt = PROMPT.format(tools=tools_text, tool_names=", ".join(TOOLS),
                               input=question, agent_scratchpad=scratchpad)
        sent.append(len(prompt))
        reply = next(replies)                                  # THOUGHT (+ action)
        if "Final Answer:" in reply:
            return reply.split("Final Answer:")[1].strip(), scratchpad, sent
        action = re.search(r"Action:\s*(.+)", reply).group(1).strip()
        arg = re.search(r"Action Input:\s*(.+)", reply).group(1).strip()
        observation = (TOOLS[action].invoke(arg) if action in TOOLS          # ACTION
                       else f"{action} is not a valid tool, try one of [{', '.join(TOOLS)}].")
        scratchpad += f"{reply}\nObservation: {observation}\nThought:"      # OBSERVATION
    return None, scratchpad, sent

answer, pad, sent = react(
    "Find the capital of Madhya Pradesh, then find its current weather.",
    [" I should first find the capital of Madhya Pradesh.\n"
     "Action: search\nAction Input: capital of Madhya Pradesh",
     " The capital is Bhopal. Now I need its weather.\n"
     "Action: get_weather_data\nAction Input: Bhopal",
     " I now know the final answer.\n"
     "Final Answer: The capital of Madhya Pradesh is Bhopal, and it is "
     "currently partly cloudy at 40 degrees."])
print(answer)
print("prompt characters per call:", sent, "total", sum(sent))
# prompt characters per call: [770, 905, 1070] total 2745
#
# Every call re-sends the whole prompt plus the growing scratchpad - lesson
# 004's buffer memory again. And the trace is readable end to end, which is
# ReAct's real selling point.

In [ ]:
assert sent == [770, 905, 1070] and sum(sent) == 2745
print(("Thought:" + pad).strip())

Every call re-sends the prompt **plus** everything learned so far — lesson
004's buffer memory again. The trace is readable end to end, which is ReAct's
real strength.

## Part C — The modern agent

In [ ]:
from langchain.agents import create_agent
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage, HumanMessage

class ScriptedModel(GenericFakeChatModel):
    """Stands in for a tool-calling model. Replays scripted replies and
    records how many messages each call was sent."""
    sent: list = []
    def bind_tools(self, tools, **kwargs):
        return self
    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        self.sent.append(len(messages))
        return super()._generate(messages, stop=stop, run_manager=run_manager, **kwargs)

model = ScriptedModel(messages=iter([
    AIMessage(content="", tool_calls=[{"name": "search",
              "args": {"query": "capital of France"}, "id": "c1"}]),
    AIMessage(content="", tool_calls=[{"name": "search",
              "args": {"query": "population of Paris"}, "id": "c2"}]),
    AIMessage(content="Paris is the capital of France, with about 2.1 million people."),
]))
model.sent = []

agent = create_agent(model=model, tools=[search, get_weather_data])
print(type(agent).__name__)                 # CompiledStateGraph - a LangGraph graph

out = agent.invoke({"messages": [HumanMessage(
    "What is the population of the capital of France?")]})
for m in out["messages"]:
    print(f"{type(m).__name__:<13}", m.content or m.tool_calls[0]["args"])
print("messages sent per model call:", model.sent)        # [1, 3, 5]
#
# The tools really ran ("Paris", "2.1 million"); the model's decisions were
# scripted. Nobody wrote the loop - the graph IS the loop. In lesson 019 the
# programmer chose which tool ran when; here the model chooses.

In [ ]:
assert type(agent).__name__ == "CompiledStateGraph"
assert model.sent == [1, 3, 5]
assert out["messages"][2].content == "Paris"          # the tool really ran

## Part D — The limit that stops a looping agent

Only explicit limits are run here. The default is read from the installed code
— running it would mean about 5,000 model calls.

In [ ]:
import inspect, re as _re
from langchain.agents import factory
from langgraph.errors import GraphRecursionError

default = int(_re.search(r'"recursion_limit":\s*([\d_]+)',
                         inspect.getsource(factory)).group(1).replace("_", ""))
print("create_agent default recursion_limit:", default)      # 9999

def forever():
    i = 0
    while True:                       # a model that never stops asking for tools
        i += 1
        yield AIMessage(content="", tool_calls=[{"name": "search",
                        "args": {"query": "capital of France"}, "id": f"c{i}"}])

for limit in (10, 25):
    m = ScriptedModel(messages=forever()); m.sent = []
    try:
        create_agent(model=m, tools=[search]).invoke(
            {"messages": [HumanMessage("What is the capital of France?")]},
            config={"recursion_limit": limit})
    except GraphRecursionError:
        print(f"limit {limit}: stopped after {len(m.sent)} model calls")
# limit 10: stopped after 5 model calls
# limit 25: stopped after 13 model calls
#
# DO NOT leave it at the default. At 9,999, a looping agent makes about
# 5,000 model calls - each paid for, each longer than the last - before
# anything stops it. Set recursion_limit yourself.

In [ ]:
assert default == 9999
print("set recursion_limit yourself")

## What to take away

- An agent decides the **sequence** itself; that is the difference from
  lesson 019.
- ReAct loops Thought → Action → Observation, and its cost grows with every
  step because the whole trace is re-sent.
- `create_agent` returns a **LangGraph graph** — the graph is the loop.
- Its default `recursion_limit` is **9,999**. Set your own.

## Exercises

1. Give the scripted ReAct model six steps instead of three. Predict the total
   characters sent before running it.
2. Make one tool raise an exception inside `create_agent`. What does the model
   see in the next message?
3. Add a `get_weather_data` step to the `create_agent` script and trace the
   messages sent per call.
4. If you have an API key, replace `ScriptedModel` with a real chat model and
   ask the Madhya Pradesh question. Does it choose the same two tools?